In [1]:
import tensorflow as tf
from pathlib import Path

In [12]:
tf.random.set_seed(123)

project_dir = Path.cwd().parent
train_dir = project_dir / "img/train"
validation_dir = project_dir / "img/validate"

img_height = 180
img_width = 180
batch_size = 32

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

class_names = train_dataset.class_names

AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

""" 
I created another notebook to give EfficientNetB0 architecture a try but there seems 
to be an issue with my versions of tf and keras as I tried many different tweaks and 
kept getting a shape mismatch error, so the MobileNetV2 architecture below seems 
like the best fit for my small dataset if I want to use transfer learning
"""
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(img_height, img_width, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
    tf.keras.layers.RandomFlip('horizontal'),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15
)

val_loss, val_acc = model.evaluate(validation_dataset, verbose=2)

model.save('../models/mobilenetv2_model.keras')

Found 161 files belonging to 3 classes.


Found 42 files belonging to 3 classes.


/var/folders/04/s_rk8gfn6vq1608hxwwtl_040000gn/T/ipykernel_20370/2627652257.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(


Epoch 1/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 7s 711ms/step - accuracy: 0.3727 - loss: 1.8224 - val_accuracy: 0.4048 - val_loss: 1.2380
Epoch 2/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.4845 - loss: 1.3167 - val_accuracy: 0.6905 - val_loss: 0.6600
Epoch 3/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 99ms/step - accuracy: 0.5963 - loss: 1.0588 - val_accuracy: 0.8095 - val_loss: 0.4212
Epoch 4/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 101ms/step - accuracy: 0.6335 - loss: 0.8227 - val_accuracy: 0.7857 - val_loss: 0.3749
Epoch 5/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 102ms/step - accuracy: 0.6832 - loss: 0.8570 - val_accuracy: 0.8810 - val_loss: 0.3208
Epoch 6/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 120ms/step - accuracy: 0.7329 - loss: 0.7826 - val_accuracy: 0.8810 - val_loss: 0.3898
Epoch 7/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 105ms/step - accuracy: 0.7453 - loss: 0.9076 - val_accuracy: 0.8333 - val_loss: 0.3069
Epoch 8/15
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 107ms/step - accuracy: 0.7205 - loss: 0.7388 - val_accuracy: 0.8810 - val_loss: 